# Household rates and reproducible cleaning

[Run in browser](https://muzammilafroz.github.io/applied-economics-data-learning-lab/lab/index.html?path=lessons/03_household_rates_cleaning.ipynb) | [Open in Colab](https://colab.research.google.com/github/muzammilafroz/applied-economics-data-learning-lab/blob/main/notebooks/lessons/03_household_rates_cleaning.ipynb) | [Course home](https://muzammilafroz.github.io/applied-economics-data-learning-lab/) | [Take the test](https://muzammilafroz.github.io/applied-economics-data-learning-lab/tests/?module=module-03)

> This independent learning resource uses fictional, synthetic data. It is not an official assessment or credential.

## Why this lesson matters

Household survey files often arrive in different shapes. One file may contain one household per row while another spreads member test results across many columns. A reproducible pipeline must define the denominator, protect identifiers, validate the merge, document special codes, and preserve provenance.

Prerequisite: Lesson 1. The data are fictional and independently simulated.

In [1]:
from __future__ import annotations

import os
import sys
import types
from urllib.request import urlopen

if sys.platform == "emscripten":
    import piplite
    await piplite.install("pyodide-http")
    import pyodide_http
    pyodide_http.patch_all()

RAW_CODE_ROOT = "https://raw.githubusercontent.com/muzammilafroz/applied-economics-data-learning-lab/v1.0.0/learning_lab"
if sys.platform == "emscripten":
    from js import window
    if window.location.hostname in {"127.0.0.1", "localhost"}:
        RAW_CODE_ROOT = f"{window.location.origin}/learning_lab"
        os.environ["LEARNING_LAB_DATA_BASE"] = f"{window.location.origin}/data/teaching"

def load_public_module(module_name):
    """Import locally, or fetch the small public helper when running in Colab/Lite."""
    try:
        return __import__(f"learning_lab.{module_name}", fromlist=[module_name])
    except ModuleNotFoundError:
        location = f"{RAW_CODE_ROOT}/{module_name}.py"
        source = urlopen(location).read().decode("utf-8")
        module = types.ModuleType(f"learning_lab.{module_name}")
        exec(compile(source, location, "exec"), module.__dict__)
        return module

lab_io = load_public_module("io")
get_data_url = lab_io.get_data_url
read_teaching_csv = lab_io.read_teaching_csv
read_teaching_geojson = lab_io.read_teaching_geojson

print("Runtime:", sys.platform)
print("Data reference:", os.getenv("LEARNING_LAB_DATA_REF", "v1.0.0"))

Runtime: win32
Data reference: v1.0.0


In [2]:
import numpy as np
import pandas as pd

transformations = load_public_module("transformations")
normalize_hhid = transformations.normalize_hhid
construct_household_rates = transformations.construct_household_rates
clean_analysis_fields = transformations.clean_analysis_fields

survey = read_teaching_csv("lumen_2018_survey.csv", dtype={"hhid": "string"})
tests = read_teaching_csv("lumen_2018_test_results.csv", dtype={"hhid": "string"})
print("survey shape:", survey.shape)
print("test shape:", tests.shape)

survey shape: (360, 15)
test shape: (360, 24)


## Find the wide test slots by rule

The prefix `test_` defines the 23 member-result columns. Selecting them by a documented naming rule is safer than typing every column name by hand. The assertion prevents a silently incomplete denominator if a slot is renamed or omitted.

In [3]:
test_columns = [column for column in tests.columns if column.startswith("test_")]
print(test_columns[:3], "...", test_columns[-3:])
assert len(test_columns) == 23, "expected exactly 23 test slots"

['test_01', 'test_02', 'test_03'] ... ['test_21', 'test_22', 'test_23']


## Define numerator and denominator before calculating

The main denominator is every nonblank observed result, including `inconclusive`. The numerator counts only `positive`. This answers: among people with any recorded test outcome, what fraction were positive?

Counting only positive and negative results would answer a different question and should be labelled as a sensitivity, not silently substituted.

In [4]:
labels = tests[test_columns].astype("string").apply(
    lambda column: column.str.strip().str.lower()
)
sampled_member_count = labels.notna().sum(axis=1)
positive_count = labels.eq("positive").sum(axis=1)
positive_rate = positive_count.div(sampled_member_count).where(sampled_member_count.gt(0))

print("Observed outcome labels:", sorted(labels.stack().unique().tolist()))
print("Undefined rates:", int(positive_rate.isna().sum()))
assert positive_count.le(sampled_member_count).all()
assert positive_rate.dropna().between(0, 1).all()

Observed outcome labels: ['inconclusive', 'negative', 'positive']
Undefined rates: 52


## Why zero denominators become missing

If no member has a recorded result, both numerator and denominator are zero. Writing a rate of zero would claim that the household was observed and had no positives. The honest value is missing because the rate is undefined.

In [5]:
zero_denominator_preview = pd.DataFrame(
    {
        "hhid": tests["hhid"],
        "denominator": sampled_member_count,
        "numerator": positive_count,
        "rate": positive_rate,
    }
).loc[sampled_member_count.eq(0)].head()
display(zero_denominator_preview)

,hhid,denominator,numerator,rate
0,04000000,0,0,<NA>
7,04000007,0,0,<NA>
14,04000014,0,0,<NA>
21,04000021,0,0,<NA>
28,04000028,0,0,<NA>


## Preserve and normalize household IDs

An ID is text even when all its characters are digits. The helper strips surrounding spaces, rejects missing or nondigit values, rejects values longer than eight positions, checks uniqueness, and pads on the left with zeros. It also retains a raw copy before normalization.

In [6]:
rates = construct_household_rates(tests)
survey = survey.assign(
    hhid_raw=survey["hhid"],
    hhid=normalize_hhid(survey["hhid"], "survey"),
)

display(rates.head(3))
print("First raw survey ID:", repr(survey.loc[0, "hhid_raw"]))
print("First normalized survey ID:", survey.loc[0, "hhid"])

,hhid_raw,hhid,sampled_member_count,positive_count,positive_rate
0,04000000,04000000,0,0,<NA>
1,04000001,04000001,2,1,0.5
2,04000002,04000002,3,0,0.0


First raw survey ID: '   04000000'
First normalized survey ID: 04000000


## Merge with an explicit contract

Both tables should have one row per normalized household ID. `validate="one_to_one"` turns that design claim into a check. The merge indicator lets us count unmatched rows before deciding what to do with them.

In [7]:
merged = survey.merge(
    rates.drop(columns="hhid_raw"),
    on="hhid",
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_survey", ""),
)

print(merged["_merge"].value_counts())
assert merged["_merge"].eq("both").all()
assert len(merged) == len(survey)
merged = merged.drop(columns="_merge")

_merge
both          360
left_only       0
right_only      0
Name: count, dtype: int64


## Clean non-destructively

The cleaning function starts with `frame.copy()`. Raw columns remain available, while new analysis columns are added.

- `wealth_raw` is divided by 100,000 to create a readable index scale.
- Water code 996 means on premises and becomes zero minutes.
- Water code 998 means unknown and becomes missing.
- Age code 98 and implausible ages become missing.
- Bednet code -99 becomes missing.
- The long river, dam, lake, pond, stream, canal, or irrigation-channel label is classified as surface water.
- `dirty` is corrected to `dirt` without changing the source file.

In [8]:
clean_2018 = clean_analysis_fields(merged)
columns_to_compare = [
    "wealth_raw", "wealth_index", "water_minutes_raw", "water_minutes",
    "head_age_raw", "head_age", "bednets_raw", "bednets",
]
display(clean_2018.loc[:8, columns_to_compare])

assert clean_2018["wealth_index"].notna().all()
assert clean_2018["poorwater"].dropna().isin([0, 1]).all()

,wealth_raw,wealth_index,water_minutes_raw,water_minutes,head_age_raw,head_age,bednets_raw,bednets
0,-74424,-0.74424,998.0,NaN,98,NaN,-99,NaN
1,3111,0.03111,13.0,13.0,31,31.0,2,2.0
2,87075,0.87075,996.0,0.0,32,32.0,3,3.0
3,-30268,-0.30268,39.0,39.0,58,58.0,2,2.0
4,85664,0.85664,50.0,50.0,21,21.0,4,4.0
5,-132536,-1.32536,43.0,43.0,29,29.0,2,2.0
6,-38649,-0.38649,25.0,25.0,48,48.0,2,2.0
7,-152647,-1.52647,43.0,43.0,32,32.0,3,3.0
8,-101969,-1.01969,32.0,32.0,38,38.0,2,2.0


## Append waves into a repeated cross-section

`pd.concat` stacks rows that share a schema. These are different households sampled in different waves, so the result is a repeated cross-section, not a household panel. A `survey_wave` label identifies each country-year source.

In [9]:
analysis = read_teaching_csv("synthetic_analysis_clean.csv", dtype={"hhid": "string"})
wave_counts = analysis["survey_wave"].value_counts().sort_index()
print(wave_counts)

assert analysis["source_record_key"].is_unique
assert analysis["positive_rate"].between(0, 1).all()
assert analysis["survey_wave"].nunique() == 4

survey_wave
Lumen 2014    320
Lumen 2018    308
Noria 2015    300
Noria 2021    340
Name: count, dtype: int64


## Provenance and common failures

`source_record_key` records the source wave and a source-row sequence. Provenance makes it possible to trace a pooled row back to the fictional source file.

Common failures include reading IDs as numbers, filling undefined rates with zero, editing raw special codes in place, merging without cardinality validation, and calling repeated cross-sections a panel.

Guided practice: calculate a sensitivity rate that excludes `inconclusive` from the denominator. Compare it with the main rate only for affected households. Explain why the main denominator still answers the documented primary question.